# Глава 5. Предобучение

В предыдущей главе мы рассмотрели архитектуру Трансофрмерной модели и два типа Трансформеров генериативный (GPT) и энкодерный (BERT). В этой главе посмотрим, как выглядит типовой пайплайн обучения таких моделей и сфокусируемся на этапе Pretrain.

В главе 3 мы расмматривали многослойную рекуррентную модель ELMO. Её авторы одними из первых описали парадигму обучения Transfer Learning: в рамках нее обучение модели подобно системе образования разделяется на базовое (pretrain) и специальное (fine-tuning). Парадигма мгновенно стала стандартом и все большие модели, претендовавшие на универсальность, стали использовать ее при обучении.

Несколькими годами позже многошаговый процесс обучения популяризовали OpenAI, выпустив свою модель ChatGPT, в рамках которой выделили три этапа обучения. Взрывной рост качества модели обеспечил популярность этого подхода и он де-факто стал стандартом.

Итого, типовой пайплайн обучения языковой модели состоит из этапов:
1. Pretraining<br>self-supervised обучение на терабайтах текста. Цель — научить модель *общему* представлению языка. Дорого (тысячи GPU-месяцев), делается редко, на выходе получается базовая модель<br><br>
2. Post-training<br>- SFT (обучение с учтелем по размеченным инструкциям)<br>- Alignment: точечная донастройка под предпочтения пользователей с помощью обучения с подкреплением (RLHF / DPO / GRPO и т.п.)

Стратегии пост-обучения SFT и RL не взаимно исключающие, а скорее дополняющие. Примеры моделей только с SFT: Mistral, Vicuna. Примеры моделецй только с RL: DeepSeek-R1-Zero. Примеры моделей и с тем и с другим: InstructGPT, Claude

Идея в том что почти всё знание модели закладывается на этапе предобучении. Post-training в основном *вытаскивает* и *причёсывает* уже усвоенные способности, а не учит новым фактам. Поэтому качество и состав предобучающих данных критично

Предобучение происходит в парадигме самообучения [self-supervised learning](https://en.wikipedia.org/wiki/Self-supervised_learning): обучающие примеры генерируются из самого текста, ручная разметка не нужна. Это позволяет условно "бесконечно" масштабироваться — данных в интернете очень много.

В зависимости от того, какая задача решается выделяют два типа

### Модель GPT

Мы требуем чтобы модель предсказывала *следующий токен*, зная все предыдущие токена документа. Это так называемая *causal* модель: делая предсказание для токена $t$ модели доступны токены $x_1, x_2, ... x_{t-1}$. Иногда называют также однонаправленной.

Функция потерь на таких предсказаниях - это обратный логарифм вероятности правильного токена. То есть обычная кросс-энтропия (negative log-likelihood), знакомая по логистической регрессии. 
$$
\mathcal{L}_{\text{LM}} = -\sum_{t=1}^{T} \log p_\theta(x_t \mid x_1, \dots, x_{t-1})
$$

Суммирование идет по всем токенам $1..T$ документа. 

Если от этой функции потерь взять экспоненту, то получаем классическую метрику качества, принятую в языковых моделях — перплексию (для подробностей см главу про классические методы):
$$
\text{PPL} = \exp\!\left(\tfrac{1}{T}\sum_t -\log p_\theta(x_t \mid x_{<t})\right)
$$

Ее грубая интерепретация: «сколько с среднем вариантов продолжнения модель рассматривает при генерации следующего токена». Меньше = лучше

Почему это работает<br>
Чтобы хорошо предсказывать следующий токен в произвольном тексте, модели приходится неявно выучить грамматику, факты, причинно-следственные связи, стиль, зачатки рассуждений. Предсказание следующего токена — это, по сути, сжатие текста, а хорошее сжатие требует модели мира. Эта установка («prediction = compression = understanding») — идеологический фундамент всей GPT-парадигмы.

### Модель BERT
Задача моделей типа BERT - не продолжение текста слева направо, а восстанавление замаскированных токенов. При этом модель видит весь контекст с обеих сторон (это так называемая *bidirectional* модель). Это обсуловило применения моделей - глубокое понимание языка. BERT не единственная релаизация, но она стала настолько цитируемой, что ее название стало именем нарицательным для обозначения энкодерных моделей

В первой версии модели BERT обучение происходило на 2 задачах:
- MLM (Masked Language Modeling).<br>Случайно выбираются ~15% токенов; из них (правило 80/10/10): 80% заменяются на `[MASK]`, 10% — на случайный токен, 10% оставляются без изменений. Модель предсказывает оригинал. Трюк 80/10/10 нужен, чтобы модель не «расслаблялась», полагаясь только на наличие `[MASK]` (которого на инференсе не будет).
- NSP (Next Sentence Prediction).<br>Подаются две последовательности `[CLS] A [SEP] B [SEP]`; модель решает, идёт ли B сразу за A. В более поздних вариантах BERT (модель RoBERTa) показали, что обучение под NSP скорее вредит, и от этой задачи отказались.

GPT оптимизирует *генеративную* задачу (хорош для порождения текста), BERT — *представленческую* (хорош для понимания, классификации, эмбеддингов). 

В противостоянии этих двух моделей трансформера безоговорочную побежу одержал GPT, поскольку генеративные сценарии оказались куда более востребованными. Тем не менее энкодерные модели BERT типа заняли свое место в нишеавых приложениях, таких как информационный поиск, ранжирование документов, семантический анализ и т.п.

__Teacher forcing__ — приём обучения авторегрессионных моделей когда на каждой позиции в качестве «предыдущих токенов» подаётся истинный контекст из обучающего текста, а не то, что нагенерировала бы сама модель. Благодаря teacher forcing все позиции последовательности обучаются *одновременно* за один проход. Каузальная маска в self-attention гарантирует, что позиция $t$ не «подглядывает» в будущее, а лоссы по всем $t$ считаются разом. Без этого пришлось бы прогонять модель токен за токеном — это убило бы эффективность обучения трансформеров.
Кроме того, Teacher Forcing дает стабильность, модель учится всегда на корректных префиксах, а не накапливает собственные ошибки, которые во время обучения.

Есть и негативный эффект exposure bias: на обучении модель всегда видела *идеальный* контекст, а на инференсе она генерирует без каких-либо ограничений, поэтому одна ошибка может «потянуть» за собой следующие, уводя в распределение, которого на обучении не было. На практике для больших языковых моделей exposure bias оказался куда менее болезненным, чем опасались, но концептуально важно понимать, что это разные режимы работы модели: режим обучения (teacher forcing) и режим генерации (free-running) — это разные режимы.

## Типовой батч

Батч для предобучения — это, как правило, тензор формы `[batch_size, sequence_length]` из целочисленных ID токенов плюс служебные тензоры.

Батч для GPT (авторегрессия)
- Входы (`input_ids`) — упакованные последовательности токенов фиксированной длины (например, 2k–8k, а в современных моделях вплоть до 128k+).
- Метки (`labels`) — те же токены, сдвинутые на одну позицию (next-token). Часто это делается прямо внутри лосса: `labels[t] = input_ids[t+1]`.
- Causal attention mask — нижнетреугольная, запрещает смотреть в будущее.
- Document packing. Чтобы не терять вычисления на паддинге, короткие документы склеивают в одну длинную последовательность, разделяя специальным токеном (например, `<|endoftext|>` / EOS). Иногда внутри пакета маской запрещают «перетекание» внимания между разными документами (document-aware masking), иногда нет — это инженерный выбор.

Концептуально: один батч GPT = кусок текста + его же сдвиг как цель. Разметки нет вообще.

Батч для BERT (MLM)
- Входы — последовательности с уже внесёнными `[MASK]` (по правилу 80/10/10) и спецтокенами `[CLS]`, `[SEP]`.
- Метки MLM — оригинальные токены, но только на замаскированных позициях (на остальных лосс не считается, обычно метка `-100`).
- Segment embeddings — индикатор «предложение A / предложение B» для пар.
- Метка NSP (в оригинальном BERT) — бинарная: «B действительно следует за A» или нет.

Концептуально: один батч BERT = испорченный текст + задание восстановить выбитые куски (и, опционально, угадать связность пар).

В GPT лосс считается по *всем* позициям (каждый токен обучает модель), а в BERT — только по ~15% замаскированных. Это одна из причин, почему MLM при равном числе токенов «менее эффективен по сигналу», и почему авторегрессия так хорошо масштабируется

---

## Параметры обучения
Поскольку обучение очень дорогое, важно настроить все гиперпараметры процесса, чтобы выжать максимум

В качестве оптимизатора почти всегда используется Adam или AdamW (AdamW = Adam с корректным decoupled weight decay). Настройка $\beta_2$ и $\epsilon$ заметно влияет на стабильность на больших масштабах<br><br>

- Очень большие батчи<br>десятки тысяч–миллионы токенов на шаг (за счёт data/tensor/pipeline-параллелизма и градиентной аккумуляции). Большой батч → более «гладкий» градиент → можно держать выше LR

- Mixed precision<br>обучение в `bf16`/`fp16` с мастер-копией весов в `fp32`; bf16 стал стандартом из-за широкого динамического диапазона<br><br>
- Регуляризация и стабилизация<br>weight decay, gradient clipping (обрезка нормы градиента), иногда z-loss на логиты, аккуратная инициализация и нормировки. Цель — пережить loss spikes (всплески лосса), которые на больших моделях случаются

### Learning rate
LR - коэффициент, на который обновление параметра на каждом шаге обучения. Статичный LR не очень эффеткивен, поэтому прибегают к значениям изменяемым по расписанию. Почти все расписания — это вариации на тему разогрев (warmup) + спад (decay)

- Warmup — линейный рост LR от 0 до пика за первые сотни–тысячи шагов. Нужен, чтобы Adam-статистики «устаканились» и не разнесли свежеинициализированную модель
- Inverse square-root — $\text{lr} \propto \frac{1}{\sqrt{\text{step}}}$ после warmup. Расписание из оригинального трансформера («Attention is All You Need»); сейчас встречается реже.
- Cosine decay — после пика LR плавно по косинусу падает до небольшой доли пика (например, до 10%) к концу обучения. Самое распространённое расписание для LLM (GPT-3, Llama и др.). Минус: нужно заранее знать общее число шагов — кривая «зашита» под конкретную длину.
- Linear decay — линейный спад вместо косинуса; близок по качеству.
- WSD (Warmup-Stable-Decay) — современная альтернатива: warmup → длинная фаза с постоянным LR → короткий резкий decay в самом конце. Главное преимущество — фаза «stable» не привязана к финальной длине, поэтому обучение легко продолжать (continual pretraining) и снимать промежуточные чекпойнты; резкий decay в конце даёт скачок качества. Популяризировано в работах вокруг MiniCPM и используется в ряде современных пайплайнов (включая линейку DeepSeek). Идея — отвязать «накопление» от «дошлифовки»

## Датасеты
Первые модели типа BERT и GPT-1/2 учились на чистых датасетах типа Wikipedia, BookCorpus или WebText. Они содержали тексты высокого качества, но имели ограниченный объём, десятки млрд токенов. Очень быстро объема этих датасетов перестало хватать. 

По разным оценкам в интернете более 5 млрд страниц, содержащих текстовые данные, а если считать страницы не индексируемые поисковиками, то это триллионы. Было бы здорово скачать их все и дать возможность моделям обучаться на всем этом многообразии когда либо созданных текстов. Задача вполне выполнимая. Это может звучать контринтуитивно, но такие данные занимают совсем не большой объем, порядка 100 TB, то есть вполне могут разместиться на домашнем компьютере.

В 2007 году стартовал подобный проект, который регулярно скачивал из интернета находящиеся в открытом доступе документы и помещал в свой архив. По сути повторял работу поисковых систем, но при этом был открытым проектом. Этот корпус документов назвали __Common Crawl__ и он до сих пор остается один из самых больших баз текстовых документов, на данных которого обучаются большие текстовые модели. По состоянию на 2026 год в нем накоплено более 300 млрд страниц. Данные хранятся в максимально сыром формате, часто просто как ответ на HTTP запрос, никакой экстракции текстового содержания или очистки от кода не делается. Качество разменивается на объем.

В 2019 году Google разрабатывали свою универсальную языковую модель T5 (о ней подробнее поговорим в следующей главе) и для нее нужны были обучающие данные. Возникла идея дешево собрать корпус качественных текстов, отфильтровав Common Crawl набором простых эвристических правил, не прибегая к использованию сложных классификаторов и тому подобному. В итоге они выкинули страницы без пунктуации, выкинули слишком короткие тексты, добавили фильтрацию по стоп-словам, удалили дубликаты предложений. Получившийся датасет назвали __C4__ (Colossal Clean Crawled Corpus) и сделали общедоступным. После всех фильтров он содержал ~170 млрд токенов и также стал базой для обучения многих языковых моделей (в том числе T5, GPT-3, LLAMA и других). Этот пример показал, что из набора сырых веб документов можно собрать качественный корпус текстов

OpenAI в 2020 году разрабатывали свою флагманскую модель GPT-3 и им также нужны были данные, но они пошли по пути умной фильтрации. Для этого они обучили классификатор, где в качестве позитивных примеров выступали тексты некоторого заведомо качественного датасета (Wiki, WebText, Books), а в качестве негативных примеров - случайные тексты из Common Crawl. Документ попадал в итоговую выборку с вероятностью пропорциональной оценке его качества, которую он получил от классификатора

Авторы датасета __The Pile__ (2020) сделали ставку на разнообразие: они включили 22 разнородных источника качественных данных (книги, статьи arXiv, код GitHub, PubMed, StackExchange и т.д.). Итоговый датасет содержал ~340 млрд токенов

__RefinedWeb__ (Falcon, 2023) — показал, что аккуратно отфильтрованного веба (без курируемых источников) достаточно, чтобы догнать/перегнать модели на «чистых» миксах

HuggingFace при сборке своего датасета __FineWeb__ подошли к проблеме исследовательски и решили провести абляционный анализ: перебрать все возможные фильтры и найти те, на которых модели достигают лучшего качества на реальных задачах. В итоге они подобрали три фильтра, применили их к Common Crawl и итоговый датасет составил 15 трлн токенов.

В целом с ростом размеров языковых моделей большие наборы данных стали необходимостью. Чтобы составлять такие датасеты, процесс фильтрации должен быть отлажен, поэтому во всех современных моделях стали делать ставку именно на фильтрацию моделями

AllenAI предложили датасет __Dolma__ на ~3 трлн токенов — открытый корпус с полностью задокументированным пайплайном. Они использовали классификатор на базе fastText для языка, а также эвристики MassiveText/C4 для качества, дедупликацию на уровнях URL/документ/абзац и фильтрацию токсичности.

Доменные мега-корпуса: __The Stack v2__ содержащий ~900 млрд токенов кода.

Разнообразие важно, но также важны пропорции. Для модели LLAMA-3 собирали свой датасет. дедупликация (по адресу, по содержанию), фильтрация, экспериментально подобрали пропорцию смесей источников (текст, код, математика, языки). Практический прием - на последних итерациях обучения скормили модели наиболее качественные данные, а learning rate гасили до 0. Прием назвали Annealing.

Многие пробовали с генерацией синтетических данных. И хотя она больше актуальна для этапа файн-тюна SFT, где требуется большое кол-во разнообразных задач, но для претрейна также используются. 
Пример - линейка моделей Phi («textbooks are all you need»), Nemotron-CC (порядка трети — синтетика) — генерация обучающих текстов другой LLM

Агрегируя примеры выше, выпишем типовой пайплайн сборки обучающего датасета. Выглядит он примерно так:
1. Извлечение текста из документа HTML/WARC (важен выбор экстрактора: WET vs `trafilatura` по WARC заметно влияет на качество)
2. Определение языка (обычно fastText) и отбор нужных языков
3. Фильтрация качества: применение эвристических фильтров (длина, доля пунктуации, доля букв, «спам-словари») + модельные классификаторы качества/образовательности
4. Дедупликация: точная и приближенная (MinHash/LSH) на уровне URL адреса / содержания документа / абзаца
5. Очистка (decontamination): удаление текстов, пересекающихся с тестовыми бенчмарками (например, по совпадению n-грамм), чтобы избежать потенциального Data Leakage
6. Удаление персональных данных, фильтры по токсичности, безопасности
7. Взвешивание типов данных (сколько кода, сколько вики, сколько веба, сколько математики)

[добавить] Качество VS объем

### Тематические датасеты DeepSeek
Рассмотрим два кейса формирования датасета для двух специализиорванных моделей DeepSeek, а именно математической модели DeepSeekMath и модели кодинга DeepSeekCoder.

Начинаем с того, что берём небольшой, но качественный корпус математических текстов, например [OpenWebMath](https://arxiv.org/pdf/2310.06786). Далее обучаем бинарный классифкатор для определения "математичности" текста. Для этого берем 500k примеров из OpenWebMath в качестве положительных примеров, а в качестве отрицательных 500k случайных веб-страниц. Затем обученный классификатор применяем к большому корпусу текстов типа URL Common Crawl (40 млрд HTML-страниц) и оставляем топ-k наиболее "математичных" документов.

Анализируем реузльтаты разметки. Если у какого-то домена более 10% страниц пометились как математические, то эксперты смотрят структуру домена и добавляют математические страницы (mathoverflow.net/questions/...) в список положительнвых примеров, таким образом расширяя обучающий датасет. Далее алгоритм повторяется, классификатор заново обучается на расширенном наборе и размечает весь веб, покрытие растет. Повторяем несколько таких циклов до получения результата и делаем дедупликацию (удаление пересечений с GSM8K/MATH и т.п.)

__DeepSeek-Coder__ - модель, ориентированная на генерацию кода. Конвейер сборки датасета похож на математическую модель.

Состав датасета для обучения: 87% исходный код / 10% англоязычный код-связанный текст (GitHub Markdown, StackExchange) / 3% китайский текст; Всего порядка 2 трлн токенов. 

Начинаем с того, что собираем код открытых репозиториев с GitHub
и базово его чистим: отсекаем слишком длинные / битые / автогенерированные файлы и т.п. Внутри каждого репозитория парсятся зависимости между файлами и файлы переупорядочиваются топологически — так, чтобы зависимость шла раньше зависящего файла
Зависимые файлы склеиваются в один обучающий пример, чтобы модель видела кросс-файловый контекст (это критично для реального проектного кода). Наконец делаем дедупликацию методом MinHash по каждому склеенному репозиторию а также удаляем файлы, пересекающиеся по n-граммам с датасетами HumanEval/MBPP/MATH/GSM8K

DeepSeek-Coder учился в двух задачх next-token, а также FIM (Fill-In-the-Middle) — документ режется на префикс/середину/суффикс, и модель учится восстанавливать середину по обоим концам

## Эффекты обучения

В классическом машинном обучении кривая обучения ведет себя предсказуемо. В случае действительно больших моделей все интеременее, так как поялвются неожиданные эффекты

(Power et al., 2022) из OpenAI заметили любопытный эффект, проявлявшийся во время обучения нейронных сеетей. Он заключался в том, что в ходе обучения модель довольно быстро переобучается, разница train и test становится максимальной, но спустя много шагов валидационная точность внезапно подскакивает — у модели как бы происходит «озарение». Эффект назвали __Grokking__ по Звездным войнам

Потребовалось время, чтобы сформулировать вазможные интерпретации эффекта. Одно из правдоподобных объяснений сформулировали (Nanda et al, 2023).  модель быстро учится запоминать, в течение какого-то времени формируется обощающий алгоритм. Как только он получен, тестовая точность начинает расти. Для того чтобы выйти на обобащаемость, должна быть регуляризация. Легко наблюдается по формуле

В случае с обучением больших языковых моделей вероятность наблюдать эффект крайне низкая из-за большого размера датасета, на котором происходит обучение Модель физически не может запомнить все примеры, ей для этого просто не хватает capacity (пример показывается по 1 разу, по 2 бита на пармаметр)

Какой вывод можно сделать - плато по метрике не означает, что модель не учится, нужно более глубкое изучение процесса.

__Emergent abilities__ (Wei et al., 2022) — так назвают способности модели, которых нет у маленьких моделей и которые начинают наблюдаться при достижении некоторого масштаба. Если нарисовать зависимость навыка от размера модели, которая обучается, то будет выглядеть как ступенька

Как и в случае с эффектом Гроккинга потребовалось время, чтобы предложить свою интерпретацию. (Schaeffer et al, 2023) объяснили эффект тем, что значительная часть «скачков» — особенность выбора метрики. Если измерять качество модели дискретной метрикой ("правильно" / "неправильно"), то кривая выглядит ступенчатой, но если взять непрерывный аналог (например, вероятность), метрика будет расти плавно и вполне предсказуемо

__Double descent__ — немонотонная зависимость ошибки от сложности модели (размера / числа эпох): ошибка на тестовых данных сначала падает, затем отскакивает и растёт, затем снова падает. Это ломает классическую U-образную форму кривой, объясняемой через баланс bias и variance

Другой часто встречающйся эффект - __Loss spikes__ — внезапные скачки лосса на больших моделях. Есть разные стратегии, в том числе снижением значения параметра learning rate, пропуск «плохих» батчей, перезапуск с предыдущего чекпойнта, нормировкой эмбеддингов/логитов, аккуратная настройка параметра инерционности $\beta_2$ у Adam оптимизатора

__Memorization__ — модель буквально запоминает части обучающих данных (важно для приватности и для честности оценки → отсюда декантаминация).

Резкое снижение learning rate в конце (особенно в WSD) и подмешивание самых качественных/доменных данных на последних шагах дают ощутимый прирост на бенчмарках

## Законы масштабирования
До 2020 года масштабирование языковых моделей диктовалось скорее интуицией, чем логикой, а цена ошибки измерялась миллионами долларов за один некорректно сконфигурированный прогон обучения. Стала очевидна потребность в некоторой формуле, которая связала бы качество модели с основными параметрами оубчения, таким как длительность обучения и объема датасета. НАличие такой формулы дало бы ответ на часто встречающиеся инженерные вопросы: 
- "как долго обучать модель?"
- "датасет какого размера нужен?"
- "что лучше, взять модель побольше или увеличить датасет?"

В поиске такой формулы возникло целое направление, получившее название LLM Scaling Laws.

__Scaling laws__ — это эмпирические степенные закономерности, показывающие, как именно качество языковой модели зависит от трех основных показателей: числа параметров модели $N$, объёма данных, на которых модель обучается $D$ и используемого бюджета для вычислений $C$ (кол-ва шагов обучения). Самое главное, что дают эти законы, они дают предсказуемость: можно по уменьшенной версии модели точно спрогнозировать, какую точность даст прогон большой модели, и заранее распределить бюджет

### Закон Kaplan
В 2020 году OpenAI опубликовали одну из первых работ по теме. Они изучили, как связаны 4 показателя: $L$ (функция потерь), $N$ (размер модели в кол-ве параметров), $D$ (размер датасета в токенах) и $C$ (объем вычислений в FLOPs). главные результаты их работы:

1) оказалось, что ошибка падает по степенному закону относительно $N$, $D$ и $C$ — гладко и предсказуемо на много порядков $sdfas$

2) при фиксированном бюджете вычислений более выгодно увеличивать модель и относительно умеренно — данные. А оптимально соблюдать соотношение: размер модели рос как $N_{\text{opt}} \propto C^{\,0.73}$. При превышении модлели ..., при превышении датасета ...

3) архитектура модели вторична, ее масштаб куда важнее

Второй тезис восприняли буквально и началась гонка размеров моделей. Вскоре вышла GPT-3, модель на 175B параметров, обучавщаяся на 300B токенов

### Закон Chinchilla
В 2022 году (Hoffmann et al, 2022) из DeepMind продолжили ислледование вопроса эффективности и решили утончить формулу Каплана. Для этого они обучили более 400 моделей при контролируемых бюджетах и вывели свой вариант формулы:

$$
L(N, D) = E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}}
$$

где $E$ — неустранимая «энтропия» данных, а два члена — вклад ограниченной ёмкости модели и ограниченных данных. Минимизация $L$ при фиксированном $C \approx 6ND$ дала вывод: модель и данные надо растить примерно одинаково, $N_{\text{opt}} \propto C^{0.5}$, $D_{\text{opt}} \propto C^{0.5}$.

Родилось знаменитое практическое правило определения оптимального размера выборки ≈ 20 токенов обучающей выборки на один параметр модели. В качестве иллюстрации авторы показали, как их модель Chinchilla-70B обученная на 1.4 трлн токенов обошла более круныую модель Gopher-280B, обученную на 300 млрд токенов при 4-кратно меньших вычислениях. Оказалось, что все прежние большие языковые модели были недообучены, слишком мало данных на из кол-во параметров.

Расхождение показателей ($0.73$ против $0.5$) оказалось во многом методологическим (Pearce et al., 2024; Porian et al., 2024): Kaplan считал не-эмбеддинговые параметры и работал на меньших масштабах; при корректном учёте эмбеддингов, FLOP-ов «головы», длины warmup и настроек оптимизатора результаты сходятся к Chinchilla. Независимая репликация (Besiroglu et al., 2024) подтвердила оптимум около 20–25 токенов на параметр. Сейчас сообщество в целом принимает формулы Chinchilla как стандарт для compute-optimal

### Мало данных

На практике объем обучающего датасета не бесконечен, если он заканчивается, приходится показывать примеры повторно. Но как это влияет на обучение? (Muennighoff et al, 2023) исследовали такие кейсы обучения при ограниченном объёме уникальных данных и обнеркжили, что повторять данные безопасно, но недолго. Примерно до 4 эпох повторные токены почти эквивалентны свежим, после чего отдача быстро падает. 

Это формализует «data wall» — упор в исчерпание качественных данных — и мотивирует синтетику и более жёсткую фильтрацию.

### Вклад качества данных
(Sorscher et al, 2022) обнаружили, что если проранжировать примеры по качеству и выкинуть некачествкенные, то ошибка модели падает быстрее степенного закона. То есть грамотный отбор обучающих данных меняет кривую масштабирования. Это дало обоснование интуитивно понимаемой принципу сборки качественных датасетов через классификаторы испольщуемому в даатсетах типа FineWeb-Edu, Phi и т.п. Главный вывод: состав и качество обучающих данных не менее важны, чем их объём.

### Учет инференса
(Sardana et al, 2023)  Chinchilla оптимизирует только стоимость обучения, но в реальности модель потом обслуживает миллиарды запросов. Если мы хотим учитывать стоимость инференса, то оптимальным решением будет брать модель меньше, но обучать её на гораздо большем числе токенов «с запасом», и потом эксплуатировать.

Именно поэтому современные релизы сознательно уходят далеко за «20 токенов/параметр»: например, модель Llama-3-8B обучали на 15 трлн токенов (в терминах Chinchilla это примерно 100 кратное «переобучение»). С точки зрения обучения это «расточительно», с точки зрения оптимальности инференса оптимально

### Другие законы
- Scaling laws для трансфера и дообучения — как эффект предобучения переносится на downstream при дообучении.
- Scaling laws для MoE (Mixture-of-Experts) — отдельные зависимости для разреженных моделей (активных vs всего параметров)
- Мультимодальные scaling laws — для VLM/контрастного обучения
- Scaling laws для данных и фильтрации — как качество фильтра сдвигает кривые
